In [ ]:
from datasets import load_dataset
from IPython.display import display

ds = load_dataset("stanfordnlp/imdb") # downloading the dataset from huggingface
ds.save_to_disk("data/imdb") # saves the dataset to the project folder

In [ ]:
display(ds)

In [ ]:
df = ds["train"].to_pandas()
display(df.head(10))

In [ ]:
df.columns # The dataset has two columns

In [ ]:
df.info()

In [ ]:
df["label"].nunique() # We have to values in the Label column
df["label"].unique() # The two values are 0 and 1

In [ ]:
df["label"].value_counts()

The dataset has half of the reviews listed as positive and half listed as negative. 0 and 1.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
display(df)
df[df["label"]==1]

In [ ]:
pd.set_option("display.max_colwidth", None)
df[df["label"]==0]

## It Seems to me that 0 corresponds with the negative reviews and 1 with the positive reviews

In [ ]:
df["length"] = df["text"].str.len() # length in number of characters
display(df[["text", "length"]])

In [ ]:
df["word_count"] = df["text"].str.split().str.len() # The length in the number of words per reviews
display(df[["text", "word_count"]])

In [ ]:
df[["text", "label"]].isna().sum() # The documentation says there are no missing values, but I check just in case.

In [ ]:
html_tag_pattern = r"<[^>]+>"
has_html_tags = df["text"].str.contains(html_tag_pattern, regex=True, na=False)
print(f"Reviews containing HTML tags: {has_html_tags.sum()}")
display(df.loc[has_html_tags, "text"].head(5))

In [ ]:
import html
import re

def clean_html(text):
    text = html.unescape(text) # Decodes for example &amp;
    text = re.sub(r"<[^>]+>", "", text) # Replaces HTML-tags with spaces
    text = re.sub(r"\s+", " ", text).strip()  # Removes extra spaces
    return text

df["clean_text"] = df["text"].apply(clean_html)

## Looking for duplicates

In [ ]:
df["text"].duplicated().sum() # Looking for duplicated reviews

We have 96 duplicates

## Removing duplicates

In [ ]:
duplicate_summary = (
    df.groupby("text", as_index=False)
    .agg(
        occurrences=("label", "size"),
        label_count=("label", "nunique"),
        labels=("label", lambda values: sorted(values.unique()))
    )
    .query("occurrences > 1")
)

duplicate_summary["text_preview"] = duplicate_summary["text"].str[:120]
display(duplicate_summary[["text_preview", "occurrences", "label_count", "labels"]])

# These duplicate reviews have consistent labels and are safe to deduplicate.
safe_duplicate_texts = duplicate_summary.loc[
    duplicate_summary["label_count"] == 1, "text"
]
rows_to_remove = (
    df["text"].duplicated(keep="first")
    & df["text"].isin(safe_duplicate_texts)
)

conflicting_duplicates = duplicate_summary[duplicate_summary["label_count"] > 1]
print(f"Duplicate text groups: {len(duplicate_summary)}")
print(f"Duplicate rows safe to remove: {rows_to_remove.sum()}")
print(f"Groups with conflicting labels (kept for review): {len(conflicting_duplicates)}")

display(df.loc[rows_to_remove, ["text", "label"]].head(10))

# Keep the original df unchanged; use df_clean for later analysis/modeling.
df_clean = df.loc[~rows_to_remove].reset_index(drop=True)
print(f"Rows before: {len(df)}; rows after: {len(df_clean)}")

In [ ]:
df["word_count"].describe()

### The training dataset contains 25,000 reviews.

The shortest review contains 10 words, while the longest contains 2,470. Reviews average approximately 234 words.

- **Median:** 174 words — half of the reviews contain fewer words, and half contain more.
- **25th percentile:** 127 words — 25% of reviews contain 127 words or fewer.
- **75th percentile:** 284 words — 75% of reviews contain 284 words or fewer.

The mean is higher than the median, which suggests that a few very long reviews are pulling the average upward.

In [ ]:
len(df_clean)

In [ ]:
df_clean.info()

## Transforming the text reviews

### TF-IDF transforms text into numbers so a model can use it. It gives each word a weight in every review. For example, the word movie often gets a smaller weight because it appears in many reviews. A more unusual word that expresses an opinion may get a higher weight.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_clean["clean_text"],
    df_clean["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_clean["label"]

 )

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Antal recensioner:", len(df_clean))
print("TF-IDF-form för träning:", X_train.shape)
print("TF-IDF-form för test:", X_test.shape)

### - Training(19923, 69195) : 19 923 reviews are being described with 69 195 different terms from the training data.

### - Test(4981, 69195): 4 981 reviews are being described with the same 69 195 terms.

In [ ]:
row_number = 0
row = X_train[row_number]

terms = vectorizer.get_feature_names_out()

review_vector = pd.DataFrame({
    "word": terms[row.indices],
    "tfidf_weight": row.data
}).sort_values("tfidf_weight", ascending=False)

print(X_train_text.iloc[row_number])  # Den rensade recensionen
display(review_vector.head(20))       # Ord och TF-IDF-vikter

## Training the baseline model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print(classification_report(y_test, baseline_predictions))

## Building and training Artificial Neural Network(ANN)

installing keras and backend of choice (tensorflow, jax or torch)

In [27]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
from keras.models import Sequential
